In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  

import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

from src.config import LABELS, SEED, set_seed
from src.data import nap_du_lieu, kiem_tra_ro_ri
from src.evaluate import do_thoi_gian_du_doan, luu_ket_qua
set_seed()

In [2]:
data = nap_du_lieu()
for k in ("train", "val", "test"):
    print(k, len(data["texts"][k]), np.bincount(data["y"][k]))
print("Văn bản trùng giữa các tập:", kiem_tra_ro_ri(data))
for c, ten in enumerate(LABELS):
    idx = np.where(data["y"]["train"] == c)[0][:5]
    print(f"\n[{ten}]")
    for i in idx:
        print("  -", data["texts"]["train"][i][:110])

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

c:\Users\ADMIN\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\datasets--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

train 108000 [27000 27000 27000 27000]
val 12000 [3000 3000 3000 3000]
test 7600 [1900 1900 1900 1900]
Văn bản trùng giữa các tập: {'train&val': 0, 'train&test': 0, 'val&test': 0}

[World]
  - Explosions caused mushroom cloud over N. Korea: source SEOUL, SOUTH KOREA - A mushroom-shaped cloud was seen i
  - Probe questions account given by British former home secretary in visa row (AFP) AFP - An official inquiry wil
  - Japan's homeless face ageism The number of Japanese homeless is on the rise, and cultural attitudes about age 
  - Asia-Pacific partners fight terror, nuclear spread, trade barriers SANTIAGO (AFP) - US Secretary of State Coli
  - Koizumi Replaces Key Ministers Prime Minister Junichiro Koizumi reshuffled his cabinet on Monday, replacing se

[Sports]
  - 10 seconds that change everything ATHENS - Ten seconds. Barely time enough to tie a shoe, wash a glass, get th
  - Charline Labonte rises to challenge Charline Labonte has served notice she wants to become the next startin

In [3]:
def tao_pipeline(C):
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LinearSVC(C=C, random_state=SEED))

Xtr, ytr = data["texts"]["train"], data["y"]["train"]
best = None
for C in (0.1, 0.3, 1, 3):
    t0 = time.perf_counter()
    pipe = tao_pipeline(C).fit(Xtr, ytr)
    dt = time.perf_counter() - t0
    acc_val = accuracy_score(data["y"]["val"], pipe.predict(data["texts"]["val"]))
    print(f"C={C}: val acc={acc_val:.4f}, fit {dt:.1f}s")
    if best is None or acc_val > best[0]:
        best = (acc_val, C, pipe, dt)
acc_val, C, pipe, train_s = best
print("Chọn C =", C)

C=0.1: val acc=0.9241, fit 17.7s
C=0.3: val acc=0.9307, fit 14.4s
C=1: val acc=0.9293, fit 15.8s
C=3: val acc=0.9253, fit 41.1s
Chọn C = 0.3


In [4]:
y_pred = pipe.predict(data["texts"]["test"])
print(classification_report(data["y"]["test"], y_pred, target_names=LABELS, digits=4))
clf = pipe.named_steps["linearsvc"]
luu_ket_qua("tfidf_svc", data["y"]["test"], y_pred, train_s, clf.coef_.size + clf.intercept_.size,
            do_thoi_gian_du_doan(lambda t: pipe.predict([t]), data["texts"]["test"]),
            extra={"val_accuracy": float(acc_val), "C": C})

              precision    recall  f1-score   support

       World     0.9419    0.9047    0.9230      1900
      Sports     0.9542    0.9868    0.9702      1900
    Business     0.8986    0.8958    0.8972      1900
    Sci/Tech     0.9019    0.9095    0.9057      1900

    accuracy                         0.9242      7600
   macro avg     0.9242    0.9242    0.9240      7600
weighted avg     0.9242    0.9242    0.9240      7600

{'model': 'tfidf_svc', 'accuracy': 0.9242105263157895, 'macro_f1': 0.9240163270031407, 'train_seconds': 14.428726399957668, 'params': 1686996, 'predict_ms': 0.7251590001396835, 'val_accuracy': 0.93075, 'C': 0.3}


{'model': 'tfidf_svc',
 'accuracy': 0.9242105263157895,
 'macro_f1': 0.9240163270031407,
 'train_seconds': 14.428726399957668,
 'params': 1686996,
 'predict_ms': 0.7251590001396835,
 'val_accuracy': 0.93075,
 'C': 0.3}